In [1]:
from implementation import (
    build_vocabulary,
    compute_counts,
    compute_tf,
    compute_idf,
    compute_tfidf,
    cosine_similarity
)

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine

In [2]:
documents = [
    "cat eats fish",
    "dog eats fish",
    "cat likes fish"
]

documents

['cat eats fish', 'dog eats fish', 'cat likes fish']

### So sánh vocabulary

In [3]:
student_vocabulary = build_vocabulary(documents)

print("Student implementation:")
print(student_vocabulary)

Student implementation:
['cat', 'dog', 'eats', 'fish', 'likes']


In [4]:
vectorizer = CountVectorizer()

X_count = vectorizer.fit_transform(documents)

sklearn_vocabulary = sorted(
    vectorizer.get_feature_names_out()
)

print("Sklearn:")
print(sklearn_vocabulary)

Sklearn:
['cat', 'dog', 'eats', 'fish', 'likes']


Nhận xét: Hai vocabulary là giống nhau.

Tuy nhiên, implementation sử dụng lower().split(), trong khi CountVectorizer sử dụng quy tắc tokenization mặc định. Vì vậy với corpus chứa dấu câu, từ ghép, ký hiệu hoặc các trường hợp đặc biệt, vocabulary có thể khác nhau.

### So sánh count matrix

In [5]:
student_counts = compute_counts(
    documents,
    student_vocabulary
)

print("Student Count Matrix:")

for row in student_counts:
    print(row)

Student Count Matrix:
[1, 0, 1, 1, 0]
[0, 1, 1, 1, 0]
[1, 0, 0, 1, 1]


In [6]:
print("Sklearn Count Matrix:")
print(X_count.toarray())

Sklearn Count Matrix:
[[1 0 1 1 0]
 [0 1 1 1 0]
 [1 0 0 1 1]]


Nhận xét:  Hai count matrix là giống nhau.

Tương tự như trên, với vocabulary trong trường hợp khác nhau cũng ra count matrix khác nhau.

### So sánh TF

In [7]:
student_tf = []

for count_vector in student_counts:
    tf = compute_tf(count_vector)
    student_tf.append(tf)

print("Student TF:")

for row in student_tf:
    print(row)

Student TF:
[0.3333333333333333, 0.0, 0.3333333333333333, 0.3333333333333333, 0.0]
[0.0, 0.3333333333333333, 0.3333333333333333, 0.3333333333333333, 0.0]
[0.3333333333333333, 0.0, 0.0, 0.3333333333333333, 0.3333333333333333]


In [8]:
sklearn_tf = []

for row in X_count.toarray():

    total = sum(row)

    tf = []

    for count in row:
        tf.append(count / total)

    sklearn_tf.append(tf)

print("TF:")
for row in sklearn_tf:
    print(row)

TF:
[np.float64(0.3333333333333333), np.float64(0.0), np.float64(0.3333333333333333), np.float64(0.3333333333333333), np.float64(0.0)]
[np.float64(0.0), np.float64(0.3333333333333333), np.float64(0.3333333333333333), np.float64(0.3333333333333333), np.float64(0.0)]
[np.float64(0.3333333333333333), np.float64(0.0), np.float64(0.0), np.float64(0.3333333333333333), np.float64(0.3333333333333333)]


Nhận xét: Cả hai đều dùng chung một công thức nên kết quả giống nhau.

### So sánh IDF

In [11]:
student_idf = compute_idf(student_counts)

print("Student IDF:")

for term, value in zip(student_vocabulary, student_idf):
    print(f"{term}: {value}")

Student IDF:
cat: 0.4054651081081644
dog: 1.0986122886681098
eats: 0.4054651081081644
fish: 0.0
likes: 1.0986122886681098


In [12]:
tfidf_vectorizer = TfidfVectorizer(
    smooth_idf=False,
    norm=None
)

tfidf_vectorizer.fit(documents)

sklearn_terms = tfidf_vectorizer.get_feature_names_out()
sklearn_idf = tfidf_vectorizer.idf_

print("Sklearn IDF:")

for term, value in zip(sklearn_terms, sklearn_idf):
    print(f"{term}: {value}")

Sklearn IDF:
cat: 1.4054651081081644
dog: 2.09861228866811
eats: 1.4054651081081644
fish: 1.0
likes: 2.09861228866811


Nhận xét: IDF có sự khác biệt do Student implementation sử dụng log(N/DF), trong khi sklearn sử dụng cách tính IDF có smoothing.

### So sánh TF-IDF

In [13]:
student_tfidf = []

for tf in student_tf:

    tfidf = compute_tfidf(
        tf,
        student_idf
    )

    student_tfidf.append(tfidf)

print("Student TF-IDF:")

for row in student_tfidf:
    print(row)

Student TF-IDF:
[0.13515503603605478, 0.0, 0.13515503603605478, 0.0, 0.0]
[0.0, 0.3662040962227032, 0.13515503603605478, 0.0, 0.0]
[0.13515503603605478, 0.0, 0.0, 0.0, 0.3662040962227032]


In [14]:
X_tfidf = tfidf_vectorizer.transform(documents)

print("Sklearn TF-IDF:")
print(X_tfidf.toarray())

Sklearn TF-IDF:
[[1.40546511 0.         1.40546511 1.         0.        ]
 [0.         2.09861229 1.40546511 1.         0.        ]
 [1.40546511 0.         0.         1.         2.09861229]]


Nhận xét: Vì IDF khác nhau nên TF-IDF cũng khác nhau (vẫn cùng một công thức)

### So sánh cosine similarity

In [15]:
D1 = "cat eats fish"
D2 = "dog eats fish"

student_similarity = cosine_similarity(
    student_tfidf[0],
    student_tfidf[1]
)

print("Student cosine similarity:")
print(student_similarity)

Student cosine similarity:
0.24482975009584623


In [16]:
sklearn_similarity = sklearn_cosine(
    X_tfidf[0],
    X_tfidf[1]
)[0][0]

print("Sklearn cosine similarity:")
print(sklearn_similarity)

Sklearn cosine similarity:
0.49225493709608925


Nhận xét: Cosine similarity có thể khác nhau tương ứng do hai TF-IDF vector khác nhau (vẫn cùng công thức cosine similarity)